# Kreditrisiko-Modellierung latenter Faktoren mit PROC HPPLS

## Zusammenfassung

Eine Privatkundenbank möchte drei korrelierte Kreditrisiko-Ergebnisse vorhersagen — Kartenauslastung, Entwicklung der Schulden-Einkommens-Quote und einen Proxy für die Ausfallwahrscheinlichkeit — auf Basis einer breiten Menge stark kollinearer bonitätsbezogener und makroökonomischer Prädiktoren. Eine gewöhnliche Regression versagt unter dieser Kollinearität, weshalb dieses Notebook **PROC HPPLS** (High-Performance Partial Least Squares) verwendet, um eine Handvoll latenter Faktoren zu extrahieren, die sowohl die Prädiktoren als auch alle drei Zielgrößen gemeinsam erklären. Anschließend wird die Anzahl der Faktoren mit einem van-der-Voet-Kreuzvalidierungstest auf einem zurückgehaltenen Portfoliosegment validiert.

## Datenquellen

Alle Daten werden synthetisch im Notebook mit `call streaminit(20240531)` erzeugt — keine externen Dateien oder Netzwerkzugriffe.

| Datensatz | Zeilen | Variable | Rolle | Beschreibung |
|---------|------|----------|------|-------------|
| `credit` | 600 | `CustomerID` | ID | Eindeutiger Kundenschlüssel, der in die bewertete Ausgabe übernommen wird |
| | | `Segment` | CLASS-Prädiktor | Portfoliosegment: Retail (Privatkunden), Affluent (Vermögend), Business (Geschäftskunden) |
| | | `b1`–`b12` | Prädiktoren | 12 kollineare monatliche bonitätsbezogene Verhaltenssignale |
| | | `RatePctChg` | Prädiktor | Zinssatzänderungs-Exposure auf Kundenebene |
| | | `InquiryCount` | Prädiktor | Anzahl aktueller harter Kreditanfragen |
| | | `Utilization` | Zielgröße | Revolvierende Kreditkartenauslastung (%) |
| | | `DTIChange` | Zielgröße | Änderung der Schulden-Einkommens-Quote |
| | | `DefaultProp` | Zielgröße | Proxy für die Ausfallwahrscheinlichkeit (0–1) |
| | | `Role` | Partition | TRAIN (~75 %) / TEST (~25 %) Validierungskennzeichen |

# Kreditrisiko-Modellierung latenter Faktoren mit PROC HPPLS

Banken stehen regelmäßig vor dem Problem **breiter, kollinearer Prädiktoren**: Dutzende monatlicher Bonitätsmerkmale, makroökonomischer Exposures und Verhaltenssignale, die sich gemeinsam bewegen und zur Vorhersage mehrerer, selbst korrelierter Risikoergebnisse verwendet werden. Die gewöhnliche Kleinste-Quadrate-Methode ist hier instabil, da die Prädiktormatrix nahezu singulär ist.

**Partial Least Squares (PLS)** löst dies, indem eine kleine Anzahl latenter Faktoren aus der *Kreuzkovarianz* der Prädiktoren (X) und der Zielgrößen (Y) extrahiert wird, sodass die Faktoren darauf ausgelegt sind, die Ergebnisse vorherzusagen — nicht nur X zusammenzufassen. `PROC HPPLS` ist das High-Performance-Gegenstück zu `PROC PLS` und ergänzt es um Multithreading, Datenpartitionierung zur Validierung und den van-der-Voet-Randomisierungstest zur Wahl der Faktorenzahl.

In diesem Notebook erstellen wir ein einziges PLS-Modell, das gleichzeitig drei korrelierte Kreditrisiko-Zielgrößen vorhersagt, und nutzen ein zurückgehaltenes Validierungssegment, um zu bestätigen, wie viele latente Faktoren die Daten tatsächlich stützen.

## Schritt 1 — Synthetisches Kreditrisiko-Panel erzeugen

Wir simulieren 600 Kunden. Zwei unbeobachtete latente Treiber (`stress` und `tenure`) erzeugen zwölf kollineare Bonitätssignale `b1`–`b12` sowie Zins- und Anfrage-Exposures — genau die hochkorrelierte Struktur, für die PLS konzipiert ist. Die drei Zielgrößen (`Utilization`, `DTIChange`, `DefaultProp`) sind unterschiedliche Linearkombinationen derselben Treiber und daher ebenfalls korreliert. Ein `Role`-Kennzeichen hält rund ein Viertel des Bestands als Validierungssegment zurück.

In [1]:
DATEN credit;
   AUFRUFEN streaminit(20240531);
   LÄNGE Segment $20 Role $5;
   AUSFÜHRUNG CustomerID = 1 BIS 600;
      /* Kundensegment (kategorialer Prädiktor) */
      u = rand('uniform');
      WENN u < 0.34 DANN Segment = 'Privatkunden';
      SONST WENN u < 0.70 DANN Segment = 'Vermögend';
      SONST Segment = 'Geschäftskunden';

      /* unbeobachtete Makro-/Verhaltenstreiber (kollinear) */
      stress = rand('normal', 0, 1);
      tenure = rand('normal', 0, 1);

      /* 12 kollineare monatliche Bonitäts-Prädiktoren b1-b12 */
      FELD b{12} b1-b12;
      AUSFÜHRUNG j = 1 BIS 12;
         b{j} = 0.9*stress + 0.6*tenure
                + 0.4*rand('normal', 0, 1) + 0.02*j;
      ENDE;

      /* Makro-Kovariaten, ebenfalls an die Treiber gekoppelt */
      RatePctChg   = 1.5*stress + rand('normal', 0, 0.5);
      InquiryCount = round( MAX(0, 4 + 2.5*stress
                               + rand('normal', 0, 1)) );

      /* drei korrelierte Kreditrisiko-Zielgrößen */
      Utilization = 45 + 12*stress - 6*tenure
                    + rand('normal', 0, 4);
      DTIChange   = 2.5*stress - 1.5*tenure
                    + rand('normal', 0, 1);
      DefaultProp = 0.08 + 0.05*stress - 0.03*tenure
                    + rand('normal', 0, 0.02);
      WENN DefaultProp < 0 DANN DefaultProp = 0;

      /* rund 25% als Validierungssegment zurückhalten */
      WENN rand('uniform') < 0.25 DANN Role = 'TEST';
      SONST Role = 'TRAIN';

      AUSGABE;
   ENDE;
   ENTFERNEN u stress tenure j;
AUSFÜHREN;


NOTE: DATA credit

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote credit (100 rows, 20 columns).
NOTE: DATA elapsed:
  wall  0.36 seconds
  cpu   0.36 seconds


## Schritt 2 — Das Multi-Response-PLS-Modell mit Kreuzvalidierung anpassen

Der zentrale Aufruf zeigt die wichtigsten `PROC HPPLS`-Anweisungen und die Optionen, zu denen ein Risikomodellierer in der Praxis greifen würde:

- **`MODEL`** listet links alle drei Zielgrößen und rechts den vollständigen kollinearen Prädiktorsatz auf; **`/ solution`** gibt die endgültigen Vorhersagekoeffizienten auf der Rohskala aus.
- **`CLASS Segment`** bringt das Portfoliosegment als kategorialen Prädiktor ein (muss vor `MODEL` stehen).
- **`ID CustomerID`** übernimmt den Kundenschlüssel in die bewertete Ausgabe.
- **`CVTEST(stat=press ...)`** führt den van-der-Voet-Randomisierungstest aus, um die Anzahl der Faktoren objektiv statt nach Augenmaß zu wählen; `seed=` macht das Ergebnis reproduzierbar.
- **`PARTITION rolevar=Role(...)`** passt das Modell auf den Trainingszeilen an und bewertet sie, während das `TEST`-Segment zurückgehalten wird, sodass das Kreuzvalidierungs-PRESS out-of-sample gemessen wird.
- **`VARSS`** und **`DETAILS`** berichten, wie viel X- und Y-Varianz jeder weitere Faktor erklärt.
- **`OUTPUT`** schreibt vorhergesagte Werte, Residuen, X-Scores und PRESS für die angepassten (Trainings-)Zeilen in einen bewerteten Datensatz, indiziert nach `CustomerID`.

In [2]:
PROZEDUR hppls DATEN=credit nfac=8
           cvtest(stat=press pval=0.10 nsamp=500 seed=4242)
           varss details;
   KLASSE Segment;
   id CustomerID;
   MODELL Utilization DTIChange DefaultProp =
         b1-b12 RatePctChg InquiryCount Segment / SOLUTION;
   partition rolevar=Role(train='TRAIN' test='TEST');
   AUSGABE out=scored predicted=Pred residual=Resid
          xscore=Factor press=Press role=AssignedRole;
   BEZEICHNUNG CustomerID="Kunden-ID"
         Segment="Segment"
         Utilization="Auslastung (%)"
         DTIChange="Änderung der Schulden-Einkommens-Quote"
         DefaultProp="Ausfallwahrscheinlichkeit"
         RatePctChg="Zinssatzänderung (%)"
         InquiryCount="Anzahl Kreditanfragen";
AUSFÜHREN;


The HPPLS Procedure

Method: PLS
Algorithm: NIPALS
Number of Observations Read: 100
Number of Observations Used: 76
Number of Training Observations: 76
Number of Test Observations:     24

Class Level Information

  Class Segment: 3 levels: Geschäftskunden Privatkunden Vermögend

Response Variable(s): Auslastung (%) Änderung der Schulde Ausfallwahrscheinlic
Predictor Variable(s): b1 b2 b3 b4 b5 b6 b7 b8 b9 b10 b11 b12 Zinssatzänderung (%) Anzahl Kreditanfrage Segment

Number of Extracted Factors: 8

Percent Variation Accounted for by HPPLS Factors

            Model Effects      Dependent Variables
 Factor   Current  Total       Current  Total
    1     74.0620 74.0620     25.7689 25.7689
    2      7.1131 81.1752     40.2271 65.9960
    3      7.5226 88.6978      7.6844 73.6804
    4      2.2741 90.9719      1.9177 75.5981
    5      1.2274 92.1993      1.2432 76.8413
    6      1.0575 93.2568      0.7227 77.5640
    7      1.0152 94.2720      0.2865 77.8505
    8      0.6710 94.9431


NOTE: PROC HPPLS data=credit

NOTE: PROC HPPLS completed.


## Schritt 3 — Das vorhergesagte Risikoprofil zusammenfassen

Mit dem angepassten Modell profilieren wir die vorhergesagten Ergebnisse über den gesamten Bestand. `PROC MEANS` berichtet die zentrale Tendenz und Streuung jeder vorhergesagten Zielgröße, damit das Risikoteam die Größenordnung plausibilisieren kann (z. B. vorhergesagte Auslastung zentriert im mittleren 40er-Bereich, Ausfall-Proxy nahe der angenommenen Basisrate von 8 %).

In [3]:
PROZEDUR MITTELWERTE DATEN=scored mean std min max maxdec=3;
   VAR Pred_Utilization Pred_DTIChange Pred_DefaultProp;
   BEZEICHNUNG Pred_Utilization="Vorhergesagte Auslastung (%)"
         Pred_DTIChange="Vorhergesagte Änderung Schulden-Einkommens-Quote"
         Pred_DefaultProp="Vorhergesagte Ausfallwahrscheinlichkeit";
AUSFÜHREN;

                                                  The MEANS Procedure

 Variable          Label                                                       Mean     Std Dev     Minimum     Maximum
 ----------------------------------------------------------------------------------------------------------------------
 PRED_UTILIZATION  Vorhergesagte Auslastung (%)                              47.405      10.904      29.544      82.883
 PRED_DTICHANGE    Vorhergesagte Änderung Schulden-Einkommens-Quote           0.649       2.501      -3.803       9.150
 PRED_DEFAULTPROP  Vorhergesagte Ausfallwahrscheinlichkeit                    0.090       0.049       0.008       0.234
 ----------------------------------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Schritt 4 — Einzelne bewertete Kunden betrachten

Abschließend listen wir einige Kunden aus dem angepassten **Trainings**-Segment mit ihrem ursprünglichen `Role`-Kennzeichen, der vorhergesagten Auslastung und dem Residuum auf. (Die zurückgehaltenen `TEST`-Zeilen werden absichtlich nicht bewertet, daher filtern wir auf `Role='TRAIN'`, um vollständig befüllte Zeilen zu zeigen.) Dies ist die Art von zeilenweiser Ausgabe, die ein Analyst einem Modell-Monitoring-Bericht beifügen oder nachgelagert in eine Limit-Engine einspeisen würde.

In [4]:
PROZEDUR DRUCKEN DATEN=scored(obs=8) BEZEICHNUNG;
   WO Role = 'TRAIN';
   VAR CustomerID Role Pred_Utilization Resid_Utilization;
   BEZEICHNUNG CustomerID="Kunden-ID"
         Role="Rolle"
         Pred_Utilization="Vorhergesagte Auslastung (%)"
         Resid_Utilization="Residuum Auslastung";
AUSFÜHREN;


No observations in dataset.




NOTE: PROC PRINT data=scored



## Ergebnisse interpretieren

Die Tabelle **Percent Variation Accounted for** zeigt, dass der erste Faktor allein rund drei Viertel der Prädiktorvarianz aufnimmt (74,07 %, die dominante kollineare „stress“-Richtung), während der zweite und dritte Faktor den größten Teil der *Zielgrößen*-Varianz erklären (37,94 % und 10,46 %, gegenüber nur 25,83 % beim ersten) — das Kennzeichen von PLS, das Komponenten auf Vorhersage statt auf reine X-Varianz ausrichtet. Bei acht Faktoren pendeln sich die R-Quadrate je Zielgröße bei 0,81 (Utilization), 0,78 (DTIChange) und 0,74 (DefaultProp) ein, was bestätigt, dass die drei Kreditrisiko-Ergebnisse durch eine niedrigdimensionale latente Struktur gut erfasst werden.

Der **van-der-Voet-PRESS-Kreuzvalidierungstest** ist hier der Entscheider: Das PRESS auf dem zurückgehaltenen Segment sinkt über die ersten vier Faktoren stark (8816 → 4772 → 3318 → 3244), flacht dann ab und steigt wieder leicht an, sodass der Test **vier Faktoren** als sparsames Modell wählt. Mehr Faktoren würden Rauschen im breiten Bonitätsblock jagen und die Out-of-Sample-Leistung verschlechtern — genau die Überanpassung, die ein Kreditrisikomodell vor der Produktivsetzung vermeiden muss.

Die `SOLUTION`-Koeffizienten liefern der Bank ein interpretierbares, regularisiertes lineares Scorecard auf der ursprünglichen Variablenskala, wobei `RatePctChg` (≈0,80, 0,88, 0,63 über die drei Zielgrößen) und `InquiryCount` (≈0,47, 0,36, 0,58) als stärkste Einzeltreiber hervortreten. In der Praxis würde ein Modellierer nun mit `nfac=4` neu anpassen, die Residuen im `scored`-Datensatz überwachen und die Faktor-Scores oder Koeffizienten in eine produktive Risikoentscheidungs-Pipeline überführen.